# Segmentation alpha: view the crop

A 500 × 500 µm square at full resolution (0.325 µm/px), centred on T3 of Mouse 2 `slide04_s3`. It's made by `make_crop.py` in this folder; `mouse02_slide04_s3_T3_500um.json` records where it came from.

In [1]:
%gui qt
from pathlib import Path

import napari
import numpy as np
import tifffile

CROP = Path("mouse02_slide04_s3_T3_500um.ome.tif")

with tifffile.TiffFile(CROP) as tf:
    image = tf.asarray()
    meta = tf.ome_metadata
px_um = float(meta.split('PhysicalSizeX="', 1)[1].split('"', 1)[0])
channels = [c.split('Name="', 1)[1].split('"', 1)[0] for c in meta.split("<Channel ")[1:]]
print(CROP.name, image.shape, image.dtype, f"{px_um} µm/px", channels)

mouse02_slide04_s3_T3_500um.ome.tif (4, 1538, 1538) uint16 0.3250074021980164 µm/px ['DAPI', 'FITC', 'TRITC', 'CY5']


In [2]:
STYLE = {  # name, colormap, visible, opacity
    "CY5":   ("NeuN (CY5)", "gray", True, 0.4),
    "TRITC": ("GFP stain (TRITC)", "green", True, 1.0),
    "DAPI":  ("DAPI", "blue", True, 1.0),
    "FITC":  ("native GFP (FITC)", "yellow", False, 1.0),
}

viewer = napari.Viewer(title=CROP.stem)
for i, ch in enumerate(channels):
    name, cmap, visible, opacity = STYLE.get(ch, (ch, "gray", False, 1.0))
    viewer.add_image(image[i], name=name, colormap=cmap, blending="additive", visible=visible,
                     opacity=opacity, contrast_limits=tuple(np.percentile(image[i], [1, 99.8])),
                     scale=(px_um, px_um), units="um")
viewer.canvas.overlays.scale_bar.visible = True

## Segmentation trial: StarDist vs Cellpose-SAM on NeuN

Both models run on the NeuN (CY5) channel only, with their default settings. Each runs in its own environment inside this folder (`.venv` for Cellpose/PyTorch, `.venv-stardist` for StarDist/TensorFlow), through `run_cellpose.py` and `run_stardist.py`. A model runs only if its result isn't saved in `results/` yet; set `RERUN = True` to redo both.

There's no hand count yet, so the table compares the models **with each other**, not against the truth. "Agree" means the same cell, by overlap (IoU ≥ 0.5).

In [3]:
import json
import subprocess

import pandas as pd
from napari.utils.colormaps import DirectLabelColormap

RERUN = False
RUNS = {  # name: (environment, script, labels, colour)
    "StarDist": (".venv-stardist", "run_stardist.py", "results/stardist_neun_labels.tif", "magenta"),
    "Cellpose-SAM": (".venv", "run_cellpose.py", "results/cellpose_neun_labels.tif", "cyan"),
}
for name, (env, script, labels_path, _) in RUNS.items():
    if RERUN or not Path(labels_path).exists():
        print(f"running {name}...")
        subprocess.run([f"{env}/bin/python", script], check=True, capture_output=True)

labels = {name: tifffile.imread(r[2]) for name, r in RUNS.items()}
info = {name: json.loads(Path(r[2].replace("_labels.tif", ".json")).read_text())
        for name, r in RUNS.items()}

In [4]:
def matched(a: np.ndarray, b: np.ndarray, iou_min: float = 0.5) -> int:
    """Cells in ``a`` with a partner in ``b`` overlapping at IoU >= iou_min (one-to-one)."""
    na, nb = int(a.max()) + 1, int(b.max()) + 1
    inter = np.bincount(a.ravel().astype(np.int64) * nb + b.ravel(), minlength=na * nb).reshape(na, nb)
    area_a, area_b = inter.sum(1), inter.sum(0)
    iou = inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)
    iou[0, :] = iou[:, 0] = 0
    return int((iou >= iou_min).any(axis=1).sum())


def sizes_um(lab: np.ndarray) -> np.ndarray:
    """Equivalent-circle diameter of each cell, in µm."""
    area = np.bincount(lab.ravel())[1:] * px_um**2
    return 2 * np.sqrt(area[area > 0] / np.pi)


rows = []
for name, lab in labels.items():
    d = sizes_um(lab)
    rows.append({"model": name, "cells": int(lab.max()),
                 "cells per mm²": round(lab.max() / (image.shape[1] * image.shape[2] * px_um**2 / 1e6)),
                 "median diameter (µm)": round(float(np.median(d)), 1),
                 "diameter 10–90% (µm)": f"{np.percentile(d, 10):.1f}–{np.percentile(d, 90):.1f}",
                 "run time (s)": info[name]["seconds"]})
display(pd.DataFrame(rows).set_index("model"))

(a_name, a), (b_name, b) = labels.items()
both = matched(a, b)
print(f"same cell found by both: {both}  ·  only {a_name}: {int(a.max()) - both}  ·  "
      f"only {b_name}: {int(b.max()) - both}  ·  agreement F1 = {2 * both / (a.max() + b.max()):.2f}")

,cells,cells per mm²,median diameter (µm),diameter 10–90% (µm),run time (s)
model,,,,,
StarDist,428,1713,15.5,9.2–18.4,2.3
Cellpose-SAM,308,1233,14.8,10.5–19.2,21.7


same cell found by both: 271  ·  only StarDist: 157  ·  only Cellpose-SAM: 37  ·  agreement F1 = 0.74


In [5]:
# Outlines on top of the image above (magenta = StarDist, cyan = Cellpose-SAM).
for name, (_, _, _, colour) in RUNS.items():
    layer = viewer.add_labels(labels[name], name=f"{name} ({int(labels[name].max())})",
                              scale=(px_um, px_um), units="um", opacity=1.0)
    layer.colormap = DirectLabelColormap(color_dict={None: colour, 0: "transparent"})
    layer.contour = 2